In [0]:
# On fusionne les deux types de données du Bronze (BATCH et STREAMING)

spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA bronze")

df_bronze = spark.table("transactions_bronze")
#df_stream = spark.table("transactions_bronze_stream")

# Harmonisation des colonnes
#common_cols = list(set(df_batch.columns).intersection(df_stream.columns))

#df_bronze = df_batch.select(common_cols).unionByName(df_stream.select(common_cols))


In [0]:
# nettoyage Siver (typage et normalisation)
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Nettoyage des valeurs nulles
    .na.drop(subset=["Amount", "Class"])
    
    # Typage
    .withColumn("Amount", F.col("amount").cast("double"))
    .withColumn("is_fraud", F.col("Class").cast("int"))
    
    # Suppression des doublons
    .dropDuplicates()

    # Suppression de la colonne Class
    .drop("Class")

    # Renommer les colonnes
    .withColumnRenamed("Amount", "amount")
    .withColumnRenamed("Time", "time")

)



In [0]:
df_silver.printSchema()

In [0]:
# Ecriture Silver dans Unit Catalog
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA silver")

df_silver.write.format("delta").mode("overwrite").saveAsTable("transactions_silver")
